# 批量计算因子教程 - 数据结构学习版

这个notebook将带你了解因子计算系统的数据结构和数据流程

## 1. 环境配置与导入

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# 添加项目根目录到Python路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from factors.registry import factor_registry
from factors.engine import FactorEngine
from storage.factor_store import FactorStore
from config.settings import get_settings
from data.providers import WindSource

print("✓ 所有模块导入成功")
print(f"项目根目录: {project_root}")

✓ 所有模块导入成功
项目根目录: d:\0信银\因子工厂


## 2. 配置参数

In [2]:
# 时间范围
START_DATE = "2024-01-01"
END_DATE = "2024-12-31"

# 股票池
DEFAULT_STOCK_POOL = [
    "600000.SH", "600036.SH", "600519.SH", "600887.SH",
    "601318.SH", "601398.SH", "601857.SH", "601939.SH",
    "601988.SH", "603259.SH",
    "000001.SZ", "000002.SZ", "000063.SZ", "000333.SZ",
    "000338.SZ", "000651.SZ", "000725.SZ", "000858.SZ",
    "002415.SZ", "300015.SZ", "300059.SZ", "300750.SZ",
]

USE_ALL_STOCKS = False
COMPUTE_CONFIG = {
    "use_cache": True,
    "skip_existing": True,
    "validate": True,
}

print(f"时间范围: {START_DATE} 至 {END_DATE}")
print(f"股票池大小: {len(DEFAULT_STOCK_POOL)}")

时间范围: 2024-01-01 至 2024-12-31
股票池大小: 22


## 3. 初始化系统组件

In [3]:
print("初始化系统组件...")

# 获取配置
settings = get_settings()
print(f"\n📋 配置信息:")
print(f"  项目根目录: {settings.project_root}")
print(f"  Wind启用: {settings.data.wind_enabled}")

# 初始化引擎
engine = FactorEngine()
print(f"\n🧮 因子引擎: {type(engine).__name__}")

# 初始化因子存储
factor_store = FactorStore()
print(f"💾 因子存储路径: {factor_store.storage_path}")

# 检查存储信息
store_info = factor_store.get_storage_info()
print(f"\n📊 存储信息:")
print(f"  文件大小: {store_info['size_mb']:.2f} MB")
print(f"  因子总数: {store_info['num_factors']}")

初始化系统组件...

📋 配置信息:
  项目根目录: d:\0信银\因子工厂
  Wind启用: True

🧮 因子引擎: FactorEngine
💾 因子存储路径: d:\0信银\因子工厂\storage\factors.h5

📊 存储信息:
  文件大小: 12.31 MB
  因子总数: 34


## 4. 查看已注册因子

In [4]:
print("📋 获取已注册因子...\n")

# 获取所有因子
all_factors = factor_registry.list_factors()
print(f"✓ 共有 {len(all_factors)} 个已注册因子\n")
print("所有因子列表:")
for i, factor in enumerate(all_factors, 1):
    print(f"  {i}. {factor}")

📋 获取已注册因子...

✓ 共有 35 个已注册因子

所有因子列表:
  1. EMA
  2. MACD
  3. RSI
  4. BOLL
  5. ATR
  6. KDJ
  7. CCI
  8. OBV
  9. RETURN
  10. MOM
  11. ACCELERATION
  12. RSQ
  13. MAX_RETURN
  14. DOWNSIDE_RISK
  15. UPSIDE_POTENTIAL
  16. RATE_OF_CHANGE
  17. WILLIAMS_R
  18. VOLUME_RATIO
  19. VOLUME_MA
  20. TURNOVER
  21. VPRICE
  22. VOLATILITY
  23. NET_FLOW
  24. VOLUME_TREND
  25. PRICE_VOLUME_TREND
  26. VOLUME_OSCILLATOR
  27. PE
  28. PB
  29. PS
  30. MARKET_CAP
  31. CIRCULATING_CAP
  32. EP
  33. BP
  34. LOG_MARKET_CAP
  35. MA


In [5]:
# 按类别查看因子
factors_by_category = factor_registry.list_factors_by_category()

print("\n📁 按类别分组的因子:\n")
for category, factors in factors_by_category.items():
    print(f"{category.upper()} ({len(factors)}个):")
    for factor in factors:
        factor_info = factor_registry.get_factor_info(factor)
        params = factor_info.get('params', {})
        params_str = ", ".join(f"{k}={v}" for k, v in params.items())
        print(f"  - {factor}: {params_str if params_str else '默认参数'}")


📁 按类别分组的因子:

TECHNICAL (9个):
  - EMA: window=20
  - MACD: fast=12, slow=26, signal=9
  - RSI: window=14
  - BOLL: window=20, num_std=2
  - ATR: window=14
  - KDJ: n=9, m1=3, m2=3
  - CCI: window=20
  - OBV: 默认参数
  - MA: window=20
MOMENTUM (9个):
  - RETURN: period=20
  - MOM: period=20
  - ACCELERATION: short_period=5, long_period=20
  - RSQ: window=20
  - MAX_RETURN: window=20
  - DOWNSIDE_RISK: window=20
  - UPSIDE_POTENTIAL: window=20
  - RATE_OF_CHANGE: period=10
  - WILLIAMS_R: window=14
VOLUME (9个):
  - VOLUME_RATIO: window=5
  - VOLUME_MA: window=20
  - TURNOVER: 默认参数
  - VPRICE: window=20
  - VOLATILITY: window=20
  - NET_FLOW: window=5
  - VOLUME_TREND: short_window=5, long_window=20
  - PRICE_VOLUME_TREND: 默认参数
  - VOLUME_OSCILLATOR: fast_window=5, slow_window=10
FUNDAMENTAL (8个):
  - PE: 默认参数
  - PB: 默认参数
  - PS: 默认参数
  - MARKET_CAP: 默认参数
  - CIRCULATING_CAP: 默认参数
  - EP: 默认参数
  - BP: 默认参数
  - LOG_MARKET_CAP: 默认参数


## 5. 连接Wind数据源

In [6]:
print("🔍 检查Wind数据源...\n")

if not settings.data.wind_enabled:
    print("✗ Wind数据源未启用")
    print("请在.env文件中设置 DATA_WIND_ENABLED=true")
else:
    print("✓ Wind数据源已启用")
    
    print("\n📡 连接Wind数据源...")
    wind_source = WindSource()
    
    if wind_source.connect():
        print("✓ Wind连接成功!")
    else:
        print("✗ Wind连接失败")

🔍 检查Wind数据源...

✓ Wind数据源已启用

📡 连接Wind数据源...
Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2024 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.
✓ Wind连接成功!


## 6. 准备股票数据

In [7]:
# 使用默认股票池
stock_list = DEFAULT_STOCK_POOL

print(f"📊 股票池: {len(stock_list)} 只股票")
print(f"股票列表: {stock_list[:5]}... (显示前5个)")

📊 股票池: 22 只股票
股票列表: ['600000.SH', '600036.SH', '600519.SH', '600887.SH', '601318.SH']... (显示前5个)


In [8]:
# 下载股票数据
print(f"\n📥 下载股票数据...")
print(f"时间范围: {START_DATE} 至 {END_DATE}")
print(f"股票数量: {len(stock_list)}\n")

data_dict = {}
success_count = 0
fail_count = 0

# 为了演示，只下载前3只股票
demo_stock_list = stock_list[:3]
print(f"演示模式：只下载前 {len(demo_stock_list)} 只股票\n")

for i, stock_code in enumerate(demo_stock_list):
    try:
        print(f"正在下载 {stock_code} ({i+1}/{len(demo_stock_list)})...", end=" ")
        
        data = wind_source.get_daily_data(
            ts_code=stock_code,
            start_date=START_DATE,
            end_date=END_DATE,
        )
        
        if not data.empty:
            data_dict[stock_code] = data
            success_count += 1
            print(f"✓ {len(data)} 条数据")
        else:
            fail_count += 1
            print("✗ 无数据")
            
    except Exception as e:
        fail_count += 1
        print(f"✗ 失败: {e}")

print(f"\n✓ 成功: {success_count} 只, ✗ 失败: {fail_count} 只")


📥 下载股票数据...
时间范围: 2024-01-01 至 2024-12-31
股票数量: 22

演示模式：只下载前 3 只股票

正在下载 600000.SH (1/3)... ✓ 242 条数据
正在下载 600036.SH (2/3)... ✓ 242 条数据
正在下载 600519.SH (3/3)... ✓ 242 条数据

✓ 成功: 3 只, ✗ 失败: 0 只


## 7. 数据结构分析：股票数据

In [9]:
print("📊 数据结构分析\n")
print(f"data_dict 类型: {type(data_dict)}")
print(f"data_dict 长度: {len(data_dict)} (股票数量)")
print(f"\n股票代码列表: {list(data_dict.keys())}")

📊 数据结构分析

data_dict 类型: <class 'dict'>
data_dict 长度: 3 (股票数量)

股票代码列表: ['600000.SH', '600036.SH', '600519.SH']


In [10]:
# 查看单只股票的数据结构
first_stock = list(data_dict.keys())[0]
first_stock_data = data_dict[first_stock]

print(f"\n📈 股票 {first_stock} 的数据结构:\n")
print(f"类型: {type(first_stock_data)}")
print(f"形状: {first_stock_data.shape} (行数, 列数)")
print(f"\n列名:")
for col in first_stock_data.columns:
    print(f"  - {col}")


📈 股票 600000.SH 的数据结构:

类型: <class 'pandas.DataFrame'>
形状: (242, 6) (行数, 列数)

列名:
  - open
  - high
  - low
  - close
  - volume
  - amount


In [11]:
# 查看数据示例
print(f"\n{first_stock} 数据示例 (前5行):\n")
print(first_stock_data.head())


600000.SH 数据示例 (前5行):

            open  high   low  close      volume       amount
trade_date                                                  
2024-01-02  6.63  6.65  6.60   6.60  22066700.0  146066304.0
2024-01-03  6.59  6.65  6.59   6.64  18203654.0  120639706.0
2024-01-04  6.64  6.67  6.55   6.62  28885978.0  190580610.0
2024-01-05  6.60  6.76  6.59   6.68  44421387.0  296976886.0
2024-01-08  6.68  6.71  6.56   6.59  37520337.0  247977825.0


In [12]:
# 查看数据类型
print(f"\n数据类型:\n")
print(first_stock_data.dtypes)


数据类型:

open      float64
high      float64
low       float64
close     float64
volume    float64
amount    float64
dtype: object


In [13]:
# 查看索引信息
print(f"\n索引信息:\n")
print(f"索引类型: {type(first_stock_data.index)}")
print(f"索引名称: {first_stock_data.index.name}")
print(f"日期范围: {first_stock_data.index.min()} 至 {first_stock_data.index.max()}")
print(f"数据点数量: {len(first_stock_data.index)}")


索引信息:

索引类型: <class 'pandas.DatetimeIndex'>
索引名称: trade_date
日期范围: 2024-01-02 00:00:00 至 2024-12-31 00:00:00
数据点数量: 242


## 8. 演示单个因子计算

In [14]:
# 选择一个简单的因子进行演示
demo_factor = "EMA"

print(f"🧮 演示计算因子: {demo_factor}\n")

# 获取因子信息
factor_info = factor_registry.get_factor_info(demo_factor)
print(f"因子信息:")
print(f"  名称: {factor_info['name']}")
print(f"  类别: {factor_info['category']}")
print(f"  描述: {factor_info['description']}")
print(f"  参数: {factor_info.get('params', {})}")

🧮 演示计算因子: EMA

因子信息:
  名称: EMA
  类别: technical
  描述: 指数移动平均线
  参数: {'window': 20}


In [15]:
# 对每只股票计算因子
print(f"\n计算因子 {demo_factor}...\n")

factor_values_dict = {}

for stock_code, stock_data in data_dict.items():
    try:
        print(f"  计算 {stock_code}...", end=" ")
        
        # 计算因子值
        factor_value = engine.compute_factor(demo_factor, stock_data)
        
        factor_values_dict[stock_code] = factor_value
        
        print(f"✓ 得到 {len(factor_value)} 个因子值")
        
    except Exception as e:
        print(f"✗ 失败: {e}")

print(f"\n✓ 成功计算 {len(factor_values_dict)} 只股票的因子值")


计算因子 EMA...

  计算 600000.SH... ✓ 得到 242 个因子值
  计算 600036.SH... ✓ 得到 242 个因子值
  计算 600519.SH... ✓ 得到 242 个因子值

✓ 成功计算 3 只股票的因子值


## 9. 数据结构分析：因子值

In [16]:
# 查看单只股票的因子值
first_stock_code = list(factor_values_dict.keys())[0]
first_factor_value = factor_values_dict[first_stock_code]

print(f"📊 单只股票的因子值结构\n")
print(f"股票: {first_stock_code}")
print(f"类型: {type(first_factor_value)}")
print(f"长度: {len(first_factor_value)}")

if isinstance(first_factor_value, pd.Series):
    print(f"\n前10个值:\n")
    print(first_factor_value.head(10))

📊 单只股票的因子值结构

股票: 600000.SH
类型: <class 'pandas.Series'>
长度: 242

前10个值:

trade_date
2024-01-02    6.600000
2024-01-03    6.603810
2024-01-04    6.605351
2024-01-05    6.612461
2024-01-08    6.610322
2024-01-09    6.610291
2024-01-10    6.606454
2024-01-11    6.599173
2024-01-12    6.589728
2024-01-15    6.582134
Name: close, dtype: float64


In [17]:
# 转换为宽表格式
print(f"\n🔄 转换为宽表格式...\n")

# 找到最长的日期序列
max_length = max(len(v) for v in factor_values_dict.values())
print(f"最长的日期序列长度: {max_length}")

# 构建DataFrame
df_data = {}
for stock_code, factor_value in factor_values_dict.items():
    # 对齐日期索引
    if len(factor_value) < max_length:
        # 填充NaN
        padding = max_length - len(factor_value)
        factor_value = pd.concat([
            pd.Series([np.nan] * padding),
            factor_value
        ], ignore_index=True)
    
    df_data[stock_code] = factor_value.values

# 使用第一只股票的日期作为索引
first_stock_factor = list(factor_values_dict.values())[0]
if hasattr(first_stock_factor, 'index'):
    index = first_stock_factor.index
else:
    # 使用第一只股票的日期
    first_stock_data = list(data_dict.values())[0]
    index = first_stock_data.index[:max_length]

factor_df = pd.DataFrame(df_data, index=index)
factor_df.index.name = 'trade_date'

print(f"✓ 转换完成")
print(f"\n宽表格式数据结构:")
print(f"  类型: {type(factor_df)}")
print(f"  形状: {factor_df.shape} (日期数, 股票数)")


🔄 转换为宽表格式...

最长的日期序列长度: 242
✓ 转换完成

宽表格式数据结构:
  类型: <class 'pandas.DataFrame'>
  形状: (242, 3) (日期数, 股票数)


In [18]:
# 查看因子DataFrame
print(f"\n因子DataFrame示例 (前10行):\n")
print(factor_df.head(10))


因子DataFrame示例 (前10行):

            600000.SH  600036.SH    600519.SH
trade_date                                   
2024-01-02   6.600000  27.580000  1685.010000
2024-01-03   6.603810  27.598095  1685.866190
2024-01-04   6.605351  27.618277  1684.259887
2024-01-05   6.612461  27.682250  1682.269421
2024-01-08   6.610322  27.724893  1678.623762
2024-01-09   6.610291  27.767284  1675.040547
2024-01-10   6.606454  27.794210  1671.846209
2024-01-11   6.599173  27.849999  1669.438951
2024-01-12   6.589728  27.915713  1666.926670
2024-01-15   6.582134  27.997074  1664.362225


In [19]:
# 查看因子数据统计
print(f"\n因子数据统计:\n")
print(factor_df.describe())


因子数据统计:

        600000.SH   600036.SH    600519.SH
count  242.000000  242.000000   242.000000
mean     8.233832   33.586798  1575.559291
std      1.096279    2.800446   104.330741
min      6.576147   27.580000  1347.267714
25%      7.134514   31.639569  1498.302035
50%      8.284789   33.457326  1560.348951
75%      9.092839   35.122177  1678.854896
max      9.976776   38.517357  1714.357299


In [20]:
# 查看缺失值情况
print(f"\n缺失值统计:\n")
missing_counts = factor_df.isnull().sum()
print(missing_counts)

print(f"\n总缺失值: {missing_counts.sum()}")
print(f"缺失率: {missing_counts.sum() / (factor_df.shape[0] * factor_df.shape[1]) * 100:.2f}%")


缺失值统计:

600000.SH    0
600036.SH    0
600519.SH    0
dtype: int64

总缺失值: 0
缺失率: 0.00%


## 10. 保存因子到存储

In [21]:
print(f"💾 保存因子到存储...\n")

# 保存因子
engine.save_factor_values(
    factor_name=demo_factor,
    factor_values=factor_df,
    params=None,
)

print("✓ 因子已保存")

💾 保存因子到存储...

✓ 因子已保存


In [22]:
# 查看保存后的存储信息
store_info = factor_store.get_storage_info()
print(f"\n更新后的存储信息:")
print(f"  文件大小: {store_info['size_mb']:.2f} MB")
print(f"  因子总数: {store_info['num_factors']}")


更新后的存储信息:
  文件大小: 12.32 MB
  因子总数: 34


In [23]:
store_info

{'storage_path': 'd:\\0信银\\因子工厂\\storage\\factors.h5',
 'exists': True,
 'size_mb': 12.31810474395752,
 'num_factors': 34,
 'factors': ['ACCELERATION_long_period_20_short_period_5',
  'ATR_window_14',
  'BOLL_num_std_2_window_20',
  'CCI_window_20',
  'DOWNSIDE_RISK_window_20',
  'EMA',
  'EMA_window_20',
  'KDJ_m1_3_m2_3_n_9',
  'MACD',
  'MACD_fast_12_signal_9_slow_26',
  'MAX_RETURN_window_20',
  'MA_window_20',
  'MOM_period_20',
  'NET_FLOW_window_5',
  'OBV',
  'PB',
  'PE',
  'PRICE_VOLUME_TREND',
  'PS',
  'RATE_OF_CHANGE_period_10',
  'RETURN_period_20',
  'RSI_window_14',
  'RSQ_window_20',
  'TURNOVER',
  'UPSIDE_POTENTIAL_window_20',
  'VOLATILITY',
  'VOLATILITY_window_20',
  'VOLUME_MA',
  'VOLUME_MA_window_20',
  'VOLUME_OSCILLATOR_fast_window_5_slow_window_10',
  'VOLUME_RATIO_window_5',
  'VOLUME_TREND_long_window_20_short_window_5',
  'VPRICE_window_20',
  'WILLIAMS_R_window_14']}

## 11. 导出因子元数据

In [24]:
print("📄 导出因子元数据...\n")

metadata_df = factor_store.export_metadata()

if not metadata_df.empty:
    print(f"元数据形状: {metadata_df.shape}")
    print(f"\n元数据列名: {list(metadata_df.columns)}")
    
    output_file = settings.project_root / "examples" / "factors_metadata.xlsx"
    metadata_df.to_excel(output_file, index=False)
    print(f"\n✓ 元数据已导出到: {output_file}")
    
    print("\n元数据预览:")
    print(metadata_df.head())

📄 导出因子元数据...

元数据形状: (34, 9)

元数据列名: ['factor_name', 'category', 'description', 'params', 'start_date', 'end_date', 'num_stocks', 'num_records', 'statistics']

✓ 元数据已导出到: d:\0信银\因子工厂\examples\factors_metadata.xlsx

元数据预览:
     factor_name category                                        description  \
0   ACCELERATION   custom  Factor ACCELERATION_long_period_20_short_period_5   
1            ATR   custom                               Factor ATR_window_14   
2           BOLL   custom                    Factor BOLL_num_std_2_window_20   
3            CCI   custom                               Factor CCI_window_20   
4  DOWNSIDE_RISK   custom                     Factor DOWNSIDE_RISK_window_20   

                                   params  start_date    end_date  num_stocks  \
0  {"short_period": 5, "long_period": 20}  2024-01-01  2024-03-29          20   
1                          {"window": 14}  1970-01-01  1970-01-01          22   
2            {"window": 20, "num_std": 2}  1970-01-01 

## 12. 数据结构总结

In [25]:
print("📊 数据结构总结\n")
print("="*50)

print("1. 股票数据 (data_dict):")
print("   类型: Dict[str, pd.DataFrame]")
print("   结构: {股票代码: 股票数据DataFrame}")
print("   DataFrame格式: 索引=日期, 列=OHLCV等字段")

print("\n2. 因子值 (factor_values_dict):")
print("   类型: Dict[str, pd.Series]")
print("   结构: {股票代码: 因子值Series}")
print("   Series格式: 索引=日期, 值=因子值")

print("\n3. 因子宽表 (factor_df):")
print("   类型: pd.DataFrame")
print("   格式: 索引=日期, 列=股票代码, 值=因子值")
print("   形状: (日期数, 股票数)")

print("\n4. 因子元数据 (metadata_df):")
print("   类型: pd.DataFrame")
print("   包含字段: factor_name, category, start_date, end_date等")

print("\n5. 存储结构 (FactorStore):")
print("   存储: HDF5格式文件")
print("   结构: /factors/{factor_name}/{params}")
print("   元数据: /metadata/{factor_name}")

📊 数据结构总结

1. 股票数据 (data_dict):
   类型: Dict[str, pd.DataFrame]
   结构: {股票代码: 股票数据DataFrame}
   DataFrame格式: 索引=日期, 列=OHLCV等字段

2. 因子值 (factor_values_dict):
   类型: Dict[str, pd.Series]
   结构: {股票代码: 因子值Series}
   Series格式: 索引=日期, 值=因子值

3. 因子宽表 (factor_df):
   类型: pd.DataFrame
   格式: 索引=日期, 列=股票代码, 值=因子值
   形状: (日期数, 股票数)

4. 因子元数据 (metadata_df):
   类型: pd.DataFrame
   包含字段: factor_name, category, start_date, end_date等

5. 存储结构 (FactorStore):
   存储: HDF5格式文件
   结构: /factors/{factor_name}/{params}
   元数据: /metadata/{factor_name}


## 13. 数据流图

In [26]:
print("🔄 数据流程:\n")
print("="*50)
print("")
print("原始数据 (Wind API)")
print("      ↓")
print("股票数据字典 {stock_code: DataFrame}")
print("      ↓")
print("因子计算 (对每只股票计算因子)")
print("      ↓")
print("因子值字典 {stock_code: Series}")
print("      ↓")
print("因子宽表 DataFrame (日期×股票)")
print("      ↓")
print("保存到HDF5文件 (/factors/{factor_name}/...)")
print("      ↓")
print("元数据记录 (/metadata/{factor_name})")

🔄 数据流程:


原始数据 (Wind API)
      ↓
股票数据字典 {stock_code: DataFrame}
      ↓
因子计算 (对每只股票计算因子)
      ↓
因子值字典 {stock_code: Series}
      ↓
因子宽表 DataFrame (日期×股票)
      ↓
保存到HDF5文件 (/factors/{factor_name}/...)
      ↓
元数据记录 (/metadata/{factor_name})


## 14. 完整流程演示 - 批量计算多个因子

In [27]:
# 选择几个不同类别的因子进行演示
demo_factors = [
    "MACD",      # 动量因子
    "VOLATILITY",    # 波动率因子
    "VOLUME_MA",   # 成交量因子
]

print(f"🧮 批量计算 {len(demo_factors)} 个因子\n")

results = {}

for factor_name in demo_factors:
    print(f"\n{'='*50}")
    print(f"计算因子: {factor_name}")
    print(f"{'='*50}")
    
    # 计算因子
    factor_values_dict = {}
    
    for stock_code, stock_data in data_dict.items():
        try:
            factor_value = engine.compute_factor(factor_name, stock_data)
            factor_values_dict[stock_code] = factor_value
        except:
            continue
    
    # 转换为宽表
    if factor_values_dict:
        max_length = max(len(v) for v in factor_values_dict.values())
        df_data = {}
        for stock_code, factor_value in factor_values_dict.items():
            if len(factor_value) < max_length:
                padding = max_length - len(factor_value)
                factor_value = pd.concat([
                    pd.Series([np.nan] * padding),
                    factor_value
                ], ignore_index=True)
            df_data[stock_code] = factor_value.values
        
        first_stock_factor = list(factor_values_dict.values())[0]
        if hasattr(first_stock_factor, 'index'):
            index = first_stock_factor.index
        else:
            first_stock_data = list(data_dict.values())[0]
            index = first_stock_data.index[:max_length]
        
        factor_df = pd.DataFrame(df_data, index=index)
        factor_df.index.name = 'trade_date'
        
        # 保存因子
        engine.save_factor_values(
            factor_name=factor_name,
            factor_values=factor_df,
            params=None,
        )
        
        results[factor_name] = factor_df
        
        print(f"\n✓ {factor_name} 计算完成")
        print(f"  形状: {factor_df.shape}")
        print(f"  均值: {factor_df.mean().mean():.4f}")
        print(f"  标准差: {factor_df.std().mean():.4f}")
    else:
        print(f"✗ {factor_name} 计算失败")

print(f"\n{'='*50}")
print(f"✓ 批量计算完成!")
print(f"成功计算 {len(results)} 个因子")

🧮 批量计算 3 个因子


计算因子: MACD

✓ MACD 计算完成
  形状: (242, 3)
  均值: -0.0064
  标准差: 6.1221

计算因子: VOLATILITY

✓ VOLATILITY 计算完成
  形状: (242, 3)
  均值: 0.4195
  标准差: 0.1297

计算因子: VOLUME_MA

✓ VOLUME_MA 计算完成
  形状: (242, 3)
  均值: 1.0149
  标准差: 0.4498

✓ 批量计算完成!
成功计算 3 个因子


## 15. 对比不同因子的数据特征

In [28]:
print("📊 对比不同因子的数据特征\n")

comparison_data = []
for factor_name, factor_df in results.items():
    comparison_data.append({
        '因子名称': factor_name,
        '形状': factor_df.shape,
        '均值': factor_df.mean().mean(),
        '标准差': factor_df.std().mean(),
        '最小值': factor_df.min().min(),
        '最大值': factor_df.max().max(),
        '缺失值数量': factor_df.isnull().sum().sum(),
        '缺失率': f"{factor_df.isnull().sum().sum() / (factor_df.shape[0] * factor_df.shape[1]) * 100:.2f}%"
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

📊 对比不同因子的数据特征

      因子名称       形状        均值      标准差        最小值        最大值  缺失值数量   缺失率
      MACD (242, 3) -0.006431 6.122105 -28.085621 100.555539      0 0.00%
VOLATILITY (242, 3)  0.419487 0.129725   0.192967   0.949411     60 8.26%
 VOLUME_MA (242, 3)  1.014934 0.449842   0.285166   3.409227     57 7.85%


## 16. 总结

In [29]:
print("🎓 学习总结\n")
print("="*50)
print("")
print("核心数据结构:")
print("1. 股票数据: Dict[str, DataFrame] - 长表格式")
print("2. 因子值: Dict[str, Series] - 每只股票的因子序列")
print("3. 因子矩阵: DataFrame - 宽表格式(日期×股票)")
print("4. 元数据: DataFrame - 因子的描述信息")
print("")
print("数据流程:")
print("原始数据 → 数据字典 → 因子计算 → 因子矩阵 → 存储")
print("")
print("存储格式:")
print("- HDF5文件存储因子矩阵")
print("- 按因子名称和参数组织")
print("- 元数据单独存储便于查询")

🎓 学习总结


核心数据结构:
1. 股票数据: Dict[str, DataFrame] - 长表格式
2. 因子值: Dict[str, Series] - 每只股票的因子序列
3. 因子矩阵: DataFrame - 宽表格式(日期×股票)
4. 元数据: DataFrame - 因子的描述信息

数据流程:
原始数据 → 数据字典 → 因子计算 → 因子矩阵 → 存储

存储格式:
- HDF5文件存储因子矩阵
- 按因子名称和参数组织
- 元数据单独存储便于查询


## 17. 清理工作

In [30]:
# 断开Wind连接
print("\n🔌 断开Wind连接...")
wind_source.disconnect()
print("✓ 已断开")

print("\n✅ 教程完成!")


🔌 断开Wind连接...
✓ 已断开

✅ 教程完成!
